# TKCE — Three-view fusion experiment (`eye_movements`)

**Model:** raw `x`  +  RF split-direction encoding  +  `tabresnet(x)`  ->  concat  ->  TabResNet  ->  prediction head (supervised).

### How to run
1. **Runtime -> Change runtime type -> GPU** (A100 on Pro+).
2. **Runtime -> Run all.**
3. When cell **4** asks, upload `openml_cache_361070.tar.gz` from your Desktop (OpenML's API is down).

Produces: a results table, **training curves** (loss + AUC), and a comparison bar chart — all downloaded by the last cell.

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> set Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 4 · upload the dataset cache bundle (OpenML API is down)
from google.colab import files
print('Upload openml_cache_361070.tar.gz from your Desktop:')
files.upload()

In [ ]:
# 5 · extract the cache
import os, glob, tarfile
hits = glob.glob('/content/**/openml_cache_361070.tar.gz', recursive=True)
assert hits, 'Upload the bundle in cell 4 first.'
dst = '/root/.cache/openml/org/openml/www'
os.makedirs(dst, exist_ok=True)
with tarfile.open(hits[0]) as t:
    t.extractall(dst)
print('extracted from', hits[0])
print('tasks:', os.listdir(dst+'/tasks'), '| datasets:', os.listdir(dst+'/datasets'))

In [ ]:
# 6 · verify the dataset loads offline
import openml
t = openml.tasks.get_task(361070, download_splits=False)
d = t.get_dataset(); X, y, *_ = d.get_data(target=d.default_target_attribute)
print('OK cached, no network:', X.shape)

In [ ]:
# 7 · RUN THE EXPERIMENT  (~10-20 min on GPU)
# --ablation also runs x-only, x+tree, x+deep so you can see which view helps.
!python -u run_fusion.py --task 361070 --epochs 800 --fusion tabresnet --device auto --ablation

In [ ]:
# 8 · show the figures: training curves (loss + AUC) and the comparison bar chart
from IPython.display import Image, display
display(Image('results/fusion/fusion_eye_movements_curves.png'))
display(Image('results/fusion/fusion_eye_movements.png'))

In [ ]:
# 9 · download everything for your professor
from google.colab import files
for f in ['fusion_eye_movements_curves.png', 'fusion_eye_movements.png',
          'fusion_eye_movements_epochs.csv', 'fusion_eye_movements.json']:
    files.download('results/fusion/'+f)